# 05 — Evaluation

**Goal.** Raw metrics tell us *how well* the model performs. Evaluation tells us *where* it struggles and *why*.

**What we cover:**
1. Residual analysis (4-panel) — time, distribution, calibration, percentage error
2. Error breakdown by segment — day of week, month, season, holiday flags
3. SHAP — what drives predictions globally and for the worst prediction
4. Bootstrap prediction intervals (80%, 95%)
5. Staffing-tier classification — the business-facing output

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

REPO = Path.cwd().parent
MODELS = REPO / 'ml-pipeline' / 'models'

cfg = json.load(open(MODELS / 'features.json'))
features = cfg['features']

engineered = pd.read_csv(REPO / 'data' / 'processed' / 'clinic_patients_engineered.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)
test_pred  = pd.read_csv(MODELS / 'test_predictions.csv', parse_dates=['date'])
cv_resid   = np.load(MODELS / 'cv_residuals.npy')

test = engineered.merge(test_pred[['date', 'pred_XGBoost']], on='date', how='inner').rename(columns={'pred_XGBoost': 'pred'})
test['residual'] = test['patients'] - test['pred']
test['abs_error'] = test['residual'].abs()
test['pct_error'] = test['residual'] / test['patients'] * 100
print(f'Test set: {len(test)} rows  |  CV residuals: n={len(cv_resid)}, std={cv_resid.std():.1f}')

## 1. Residual analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

ax = axes[0, 0]
ax.plot(test['date'], test['residual'], color='#2c3e50')
ax.axhline(0, color='black', linewidth=0.5)
ax.fill_between(test['date'], test['residual'], 0, where=(test['residual'] >= 0), color='#27ae60', alpha=0.3, label='Under-predict')
ax.fill_between(test['date'], test['residual'], 0, where=(test['residual'] < 0), color='#c0392b', alpha=0.3, label='Over-predict')
ax.set_title('Residuals over time')
ax.set_ylabel('Actual − predicted')
ax.legend()

ax = axes[0, 1]
ax.hist(test['residual'], bins=25, color='#34495e', alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.5)
ax.axvline(test['residual'].mean(), color='#c0392b', linestyle='--', label=f"Mean {test['residual'].mean():.1f}")
ax.set_title('Residual distribution')
ax.legend()

ax = axes[1, 0]
ax.scatter(test['pred'], test['patients'], alpha=0.5, color='#3498db')
lo = min(test['pred'].min(), test['patients'].min())
hi = max(test['pred'].max(), test['patients'].max())
ax.plot([lo, hi], [lo, hi], color='#c0392b', linestyle='--')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Calibration — predicted vs actual')

ax = axes[1, 1]
ax.hist(test['pct_error'], bins=25, color='#27ae60', alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title(f"Percentage errors — {test['pct_error'].abs().mean():.1f}% MAPE")
ax.set_xlabel('% error')
plt.tight_layout(); plt.show()

## 2. Error breakdown by segment

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

axes[0,0].bar(dow_order, test.groupby('day_name')['abs_error'].mean().reindex(dow_order).values, color='#2c3e50')
axes[0,0].set_title('MAE by day of week'); axes[0,0].tick_params(axis='x', rotation=30); axes[0,0].set_ylabel('MAE')

monthly = test.groupby('month')['abs_error'].mean()
axes[0,1].bar(monthly.index, monthly.values, color='#2c3e50')
axes[0,1].set_title('MAE by month'); axes[0,1].set_xlabel('Month'); axes[0,1].set_ylabel('MAE')

season_order = ['Summer','Autumn','Winter','Spring']
season_mae = test.groupby('season')['abs_error'].mean().reindex(season_order)
axes[0,2].bar(season_order, season_mae.values, color='#2c3e50')
axes[0,2].set_title('MAE by season'); axes[0,2].set_ylabel('MAE')

for ax, col, lab in zip(axes[1], ['is_public_holiday','is_flu_peak','is_day_after_public_holiday'],
                         ['Public holiday','Flu peak','Day after public holiday']):
    g = test.groupby(col)['abs_error'].mean()
    ax.bar(['Off','On'], [g.get(0, np.nan), g.get(1, np.nan)], color=['#95a5a6','#c0392b'])
    ax.set_title(f'MAE — {lab}'); ax.set_ylabel('MAE')

plt.tight_layout(); plt.show()

## 3. SHAP — what drives predictions

In [ ]:
import shap

xgb_model = xgb.XGBRegressor()
xgb_model.load_model(MODELS / 'xgboost.json')

X_test = test[features]
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, max_display=20, show=False)
plt.gcf().set_size_inches(10, 8)
plt.tight_layout(); plt.show()

In [ ]:
# Worst prediction day
worst_idx = test['abs_error'].idxmax()
worst_pos = test.index.get_loc(worst_idx)
worst = test.loc[worst_idx]
print(f"Worst day: {worst['date'].date()}")
print(f"Actual: {worst['patients']}  Predicted: {worst['pred']:.0f}  Error: {worst['residual']:.0f}")

shap.waterfall_plot(shap.Explanation(
    values=shap_values[worst_pos],
    base_values=explainer.expected_value,
    data=X_test.iloc[worst_pos].values,
    feature_names=features,
), max_display=15, show=False)
plt.tight_layout(); plt.show()

## 4. Bootstrap prediction intervals

Point predictions aren't enough — staffing decisions need ranges. We bootstrap **CV-held-out residuals** (not in-sample residuals, which are nearly zero for a gradient-boosted model) to build 80% and 95% intervals.

In [ ]:
np.random.seed(42)
n_bootstrap = 500
base_pred = test['pred'].values
bootstrap_preds = np.array([
    base_pred + np.random.choice(cv_resid, size=len(base_pred), replace=True)
    for _ in range(n_bootstrap)
])
lower_80 = np.percentile(bootstrap_preds, 10, axis=0)
upper_80 = np.percentile(bootstrap_preds, 90, axis=0)
lower_95 = np.percentile(bootstrap_preds, 2.5, axis=0)
upper_95 = np.percentile(bootstrap_preds, 97.5, axis=0)

in_80 = ((test['patients'] >= lower_80) & (test['patients'] <= upper_80)).mean()
in_95 = ((test['patients'] >= lower_95) & (test['patients'] <= upper_95)).mean()
print(f'Coverage — 80% target: {in_80:.1%}   95% target: {in_95:.1%}')

fig, ax = plt.subplots(figsize=(15, 5))
ax.fill_between(test['date'], lower_95, upper_95, alpha=0.2, color='#3498db', label='95% interval')
ax.fill_between(test['date'], lower_80, upper_80, alpha=0.35, color='#3498db', label='80% interval')
ax.plot(test['date'], test['patients'], color='black', linewidth=1.5, label='Actual')
ax.plot(test['date'], test['pred'], color='#c0392b', linewidth=1.5, linestyle='--', label='Predicted')
ax.set_title(f'XGBoost predictions with bootstrap intervals (coverage: 80→{in_80:.0%}, 95→{in_95:.0%})')
ax.set_ylabel('Patients')
ax.legend()
plt.tight_layout(); plt.show()

## 5. Staffing tier classification

The business-facing output. Ops teams don't act on "predicted 143 patients" — they act on "Medium tier → send the Medium plan."

- **Low** (<80): 2 doctors, 3 nurses — standard roster
- **Medium** (80–150): 3 doctors, 4 nurses, extra triage
- **High** (>150): surge — 4+ doctors, ED overflow coordination

In [ ]:
from sklearn.metrics import confusion_matrix

def tier(x):
    if x < 80: return 'Low'
    if x <= 150: return 'Medium'
    return 'High'

test['tier_actual'] = test['patients'].apply(tier)
test['tier_pred']   = test['pred'].apply(tier)

labels = ['Low', 'Medium', 'High']
cm = confusion_matrix(test['tier_actual'], test['tier_pred'], labels=labels)
accuracy = (test['tier_actual'] == test['tier_pred']).mean()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel('Predicted tier'); ax.set_ylabel('Actual tier')
ax.set_title(f'Staffing tier — overall accuracy {accuracy:.1%}')
plt.tight_layout(); plt.show()

print(f'\nTier distribution (actual): {test["tier_actual"].value_counts().to_dict()}')
print(f'Tier distribution (pred)  : {test["tier_pred"].value_counts().to_dict()}')

## Takeaways

- XGBoost hits MAPE ~12%, well inside the <15% target.
- Slight negative bias — model under-predicts on average, which is the conservative direction for staffing.
- Highest errors concentrate on public holidays and flu-peak weeks — the genuinely unpredictable days.
- SHAP confirms the model leans on rolling-mean features, flu-peak, and day-of-week — domain-sensible.
- Bootstrap intervals give ops teams a defensible range to plan around, not a false-precision single number.
- Tier classification turns a regression output into a direct staffing decision.